# Streaming video services: feature-leakage audit

**Owner:** Winston. Part of the netleak feature-leakage audit (see `README.md`).

**Run:** *Restart kernel and run all*. Every step is cached or skipped when its output is already current, so a re-run only recomputes what changed. Rung definitions live in `src/netleak/rungs.py`: do not filter features in this notebook.

In [ ]:
import logging
from dataclasses import asdict

import pandas as pd
from IPython.display import Image, display

from netleak import datasets, features, plots, runner
from netleak.download import download
from netleak.load import inspect_dataset
from netleak.paths import default_paths
from netleak.rungs import RUNGS

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%H:%M:%S')
paths = default_paths()
spec = datasets.get('video_services')
spec

## 1. Data

Download (skipped if present), extract features (skipped if cached), and summarise classes and hosts. Hosts per class decides which splits are meaningful (`spec.splits`).

In [ ]:
source = download(spec, paths)
features.build(spec, source, paths.cache)
summary = inspect_dataset(spec, source)
print({k: v for k, v in summary.items() if k != 'per_class'})
summary['per_class']

## 2. The restriction ladder

In [ ]:
pd.DataFrame([(r.name, r.description, ', '.join(sorted(r.drop_fields)) or '-') for r in RUNGS.values()],
             columns=['rung', 'description', 'removed fields'])

## 3. Grid: rung × model × split

Results are written to `results/` and skipped on re-run if they are current.

In [ ]:
results = pd.DataFrame([asdict(r) for r in runner.grid(spec, paths=paths)])
results.pivot_table(index=['split', 'rung'], columns='model', values='balanced_accuracy').round(3)

In [ ]:
results[['split', 'rung', 'model', 'n_features', 'macro_f1', 'fit_seconds', 'extract_seconds_per_sample']]

## 4. Which fields carry the surviving signal at R1?

In [ ]:
importance = runner.importance(spec, 'R1', 'lgbm', paths=paths)
importance.head(15)

## 5. Figures

In [ ]:
figures = plots.all_figures(paths)
for path in figures:
    if path.name in ('headline.png', 'cost.png') or spec.name in path.name:
        display(Image(filename=str(path), width=900))

## 6. Interpretation

*TODO (Winston):* what survives from R0 to R3, how the split kind changes it, and how the numbers compare with the leaderboard.